# Untuned GPT-5.4-mini semantic feature adjudication baseline

This workflow preserves the deterministic matcher, fixed prompt, v0.1 contract, and label-leakage barrier. Exact matches bypass the LLM; every LLM-adjudicated case requires user review.

In [ ]:
from pathlib import Path

from IPython.display import display

from functions.feature_matching import load_kb, load_terminology
from functions.feature_matching_validation import load_validation_dataset
from functions.llm_semantic_adjudication import (
    CONTRACT_VERSION_V0_1, DEFAULT_MODEL, OpenAISemanticAdjudicator,
    PROMPT_VERSION_V0_1, configured_model,
    create_openai_client, load_adjudication_prompt,
)
from functions.llm_semantic_adjudication_validation import (
    adjudication_error_tables, adjudication_metrics,
    attach_adjudication_ground_truth, combined_error_export,
    known_difficult_cases, run_adjudication_predictions,
)

HERE = Path.cwd()
kb = load_kb(HERE / 'kb' / 'pd_directionality_kb_v0_3.yaml')
terminology = load_terminology(HERE / 'kb' / 'credit_risk_abbreviations_v0_2.yaml')
prompt = load_adjudication_prompt(
    HERE / 'prompts' / 'semantic_feature_adjudication_v0_1.txt'
)
validation = load_validation_dataset(
    HERE / 'inputs' / 'feature_matching_validation_v0_1.csv'
)
model = configured_model(HERE / '.env')
assert model == DEFAULT_MODEL, f'Expected {DEFAULT_MODEL}, found {model}'
client = create_openai_client(HERE / '.env')
adjudicator = OpenAISemanticAdjudicator(
    client=client, prompt=prompt, model=model,
    prompt_version=PROMPT_VERSION_V0_1,
    contract_version=CONTRACT_VERSION_V0_1,
)

## Inference with the leakage barrier

The prediction function projects the frame to `feature_name` and `description`. Labels are joined only after every matcher and LLM call has completed.

In [ ]:
def show_progress(index, total, feature_name, adjudication_used):
    route = 'LLM' if adjudication_used else 'exact bypass'
    print(f'[{index:02d}/{total:02d}] {feature_name}: {route}')

predictions = run_adjudication_predictions(
    validation, kb, terminology, adjudicator, progress=show_progress
)
assert not any(column.startswith('expected_') for column in predictions.columns)
evaluation = attach_adjudication_ground_truth(predictions, validation)

In [ ]:
metrics = adjudication_metrics(evaluation)
display(metrics)

In [ ]:
display(metrics[metrics['metric'].isin([
    'expected_no_match_rejection_accuracy',
    'not_directional_accuracy',
    'insufficient_context_accuracy',
    'orientation_accuracy',
    'contract_compliance_rate',
    'invalid_structured_output_count',
    'invented_candidate_count',
])])

In [ ]:
display(known_difficult_cases(evaluation))

In [ ]:
error_tables = adjudication_error_tables(evaluation)
for table_name, table in error_tables.items():
    print(f'{table_name}: {len(table)} row(s)')
    display(table)

In [ ]:
RESULTS_PATH = HERE / 'output' / 'llm_semantic_adjudication_results_v0_1_kb_v0_3.csv'
METRICS_PATH = HERE / 'output' / 'llm_semantic_adjudication_metrics_v0_1.csv'
ERRORS_PATH = HERE / 'output' / 'llm_semantic_adjudication_errors_v0_1.csv'
evaluation.to_csv(RESULTS_PATH, index=False)
metrics.to_csv(METRICS_PATH, index=False)
combined_error_export(error_tables).to_csv(ERRORS_PATH, index=False)
print(f'Wrote {len(evaluation)} review rows to {RESULTS_PATH}')